<a href="https://colab.research.google.com/github/baramitha/bt4222/blob/main/Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Loading


In [ ]:
import pandas as pd

import pandas as pd

anime_clean = pd.read_csv('/content/anime_clean.csv')
users_sampled = pd.read_csv('/content/users_sampled_engineered.csv')
ratings_sampled = pd.read_csv('/content/ratings_sampled_engineered.csv')

# **Data Preparation**

In [17]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [20]:
model_data = ratings_sampled[['user_id', 'anime_id', 'rating']].copy()
model_data = model_data.rename(columns={'anime_id': 'movie_id'})
model_data = model_data[model_data['rating'] >0].reset_index(drop=True)
# 20% subsample for baseline
model_data = model_data.sample(frac=0.2, random_state=42).reset_index(drop=True)

# Re-encode IDs
user_encoder= LabelEncoder()
movie_encoder = LabelEncoder()
model_data['user_id']  = user_encoder.fit_transform(model_data['user_id'])
model_data['movie_id'] = movie_encoder.fit_transform(model_data['movie_id'])

# Split: 80% train, 10% val, 10% test
train_data, temp_data = train_test_split(model_data, test_size=0.2, random_state=42)
val_data, test_data= train_test_split(temp_data,  test_size=0.5, random_state=42)

train_data = train_data.reset_index(drop=True)
val_data= val_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

# Filter val and test to seen users/items (from train only)
train_users = set(train_data['user_id'].unique())
train_items = set(train_data['movie_id'].unique())

filtered_val_data = val_data[
    val_data['user_id'].isin(train_users) & val_data['movie_id'].isin(train_items)
].reset_index(drop=True)

filtered_test_data = test_data[
    test_data['user_id'].isin(train_users) & test_data['movie_id'].isin(train_items)
].reset_index(drop=True)
num_users  = len(user_encoder.classes_)
num_movies = len(movie_encoder.classes_)
print(f'Train: {len(train_data):,} | Val: {len(filtered_val_data):,} | Test: {len(filtered_test_data):,}')
print(f'Users: {num_users:,} | Anime: {num_movies:,}')
print(f'Rating range: {model_data["rating"].min():.0f}–{model_data["rating"].max():.0f}')

Train: 3,677,446 | Val: 458,037 | Test: 458,015
Users: 163,930 | Anime: 15,061
Rating range: 1–10


# **NMF without features**

In [28]:
class NCF(nn.Module):
    def __init__(self, num_users, num_movies, embedding_dim):
        super(NCF, self).__init__()
        self.user_embeddings_gmf = nn.Embedding(num_users, embedding_dim)
        self.movie_embeddings_gmf = nn.Embedding(num_movies, embedding_dim)
        self.user_embeddings_mlp = nn.Embedding(num_users, embedding_dim)
        self.movie_embeddings_mlp = nn.Embedding(num_movies,embedding_dim)
        self.fc1 = nn.Linear(2 * embedding_dim, 128)
        self.fc2 = nn.Linear(128, embedding_dim)
        self.fc_final = nn.Linear(embedding_dim, 1)
    def forward(self, user_id, movie_id):
        gmf_output = self.user_embeddings_gmf(user_id) * self.movie_embeddings_gmf(movie_id)
        mlp_input= torch.cat([self.user_embeddings_mlp(user_id),self.movie_embeddings_mlp(movie_id)], dim=-1)
        mlp_output = torch.relu(self.fc1(mlp_input))
        mlp_output = torch.relu(self.fc2(mlp_output))
        out= self.fc_final(gmf_output + mlp_output)
        return torch.clamp(out, 1, 10).squeeze()
embedding_dim = 16
model= NCF(num_users, num_movies, embedding_dim).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(f'Model device: {next(model.parameters()).device}')
# Training
best_loss= float('inf')
patience_counter= 2
patience_counter = 0
num_epochs= 10
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        user_id  = batch['user_id']
        movie_id = batch['movie_id']
        ratings  = batch['rating']
        optimizer.zero_grad()
        loss = criterion(model(user_id, movie_id), ratings)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_train_loss = epoch_loss/len(train_loader)
    print(f'Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f}', end=' | ')
    model.eval()
    eval_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            user_id = batch['user_id']
            movie_id = batch['movie_id']
            ratings = batch['rating']
            eval_loss += criterion(model(user_id, movie_id), ratings).item()
    avg_eval_loss = eval_loss/len(val_loader)
    print(f'Val Loss: {avg_eval_loss:.4f}')
    if avg_eval_loss < best_loss:
        best_loss        = avg_eval_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_neumf_model.pt')
        print(f' Best model saved (loss: {best_loss:.4f})')
    else:
        patience_counter += 1
        print(f'  No improvement. Patience: {patience_counter}/{patience}')
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break
model.load_state_dict(torch.load('best_neumf_model.pt', map_location=device))

Model device: cuda:0
Epoch 1/10 | Train Loss: 4.0821 | Val Loss: 2.5599
 Best model saved (loss: 2.5599)
Epoch 2/10 | Train Loss: 2.3435 | Val Loss: 2.1917
 Best model saved (loss: 2.1917)
Epoch 3/10 | Train Loss: 2.0010 | Val Loss: 1.9750
 Best model saved (loss: 1.9750)
Epoch 4/10 | Train Loss: 1.8146 | Val Loss: 1.8812
 Best model saved (loss: 1.8812)
Epoch 5/10 | Train Loss: 1.7185 | Val Loss: 1.8385
 Best model saved (loss: 1.8385)
Epoch 6/10 | Train Loss: 1.6602 | Val Loss: 1.8220
 Best model saved (loss: 1.8220)
Epoch 7/10 | Train Loss: 1.6166 | Val Loss: 1.8142
 Best model saved (loss: 1.8142)
Epoch 8/10 | Train Loss: 1.5799 | Val Loss: 1.8198
  No improvement. Patience: 1/2
Epoch 9/10 | Train Loss: 1.5457 | Val Loss: 1.8360
  No improvement. Patience: 2/2
Early stopping at epoch 9


<All keys matched successfully>

In [29]:
model = NCF(num_users, num_movies, embedding_dim).to(device)
model.load_state_dict(torch.load('best_neumf_model.pt', map_location=device))
model.eval()

NCF(
  (user_embeddings_gmf): Embedding(163930, 16)
  (movie_embeddings_gmf): Embedding(15061, 16)
  (user_embeddings_mlp): Embedding(163930, 16)
  (movie_embeddings_mlp): Embedding(15061, 16)
  (fc1): Linear(in_features=32, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=16, bias=True)
  (fc_final): Linear(in_features=16, out_features=1, bias=True)
)

In [30]:
def generate_predictions(model, data_loader):
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in data_loader:
            user_id  = batch['user_id']
            movie_id = batch['movie_id']
            outputs  = model(user_id, movie_id)
            predictions.extend(zip(user_id.cpu().numpy(), movie_id.cpu().numpy(), outputs.cpu().numpy()))
    return predictions
test_predictions = generate_predictions(model, test_loader)
neumf_predictions = test_predictions

In [31]:
test_predictions

[(np.int64(66555), np.int64(4438), np.float32(7.5607696)),
 (np.int64(2596), np.int64(9130), np.float32(9.875182)),
 (np.int64(7591), np.int64(6654), np.float32(7.3500338)),
 (np.int64(90289), np.int64(8096), np.float32(6.86408)),
 (np.int64(70212), np.int64(14326), np.float32(8.601202)),
 (np.int64(106083), np.int64(246), np.float32(5.524396)),
 (np.int64(33741), np.int64(8682), np.float32(6.2310085)),
 (np.int64(99304), np.int64(13020), np.float32(5.177252)),
 (np.int64(92606), np.int64(116), np.float32(9.36162)),
 (np.int64(146095), np.int64(3792), np.float32(6.1082807)),
 (np.int64(86716), np.int64(728), np.float32(9.213505)),
 (np.int64(3463), np.int64(2199), np.float32(6.9707317)),
 (np.int64(153970), np.int64(6682), np.float32(8.149221)),
 (np.int64(98428), np.int64(12067), np.float32(8.418612)),
 (np.int64(151585), np.int64(6178), np.float32(6.7816873)),
 (np.int64(139483), np.int64(8386), np.float32(5.7316127)),
 (np.int64(102239), np.int64(14540), np.float32(7.520008)),
 (np.

In [32]:
# Convert test data to DataFrame for easier manipulation
test_data_df = pd.DataFrame(filtered_test_data, columns=['user_id', 'movie_id', 'rating'])
# Map actual ratings to relevance scores
test_data_df['relevance'] = test_data_df['rating'].apply(lambda x: x if x > 0 else 0)  # Example relevance mapping
test_data_df
len(test_data_df)
len(test_predictions)

458015

In [33]:
from collections import defaultdict

def calculate_rmse(predictions, actuals):
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    return np.sqrt(np.mean((predictions - actuals) ** 2))
def calculate_mae(predictions, actuals):
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    return np.mean(np.abs(predictions - actuals))
def dcg_at_k(relevance_scores, k):
    relevance_scores = np.asarray(relevance_scores)[:k]
    if relevance_scores.size:
        return np.sum((2**relevance_scores - 1)/np.log2(np.arange(2, relevance_scores.size + 2)))
    return 0.0

def ndcg_at_k(relevance_scores, ideal_relevance_scores, k):
    dcg = dcg_at_k(relevance_scores, k)
    idcg = dcg_at_k(ideal_relevance_scores, k)
    if idcg == 0:
        return 0.0
    return dcg/idcg
def precision_at_k(recommended_items, relevant_items, k):
    recommended_at_k = set(recommended_items[:k])
    relevant_set = set(relevant_items)
    hits = len(recommended_at_k & relevant_set)
    return hits/k if k >0 else 0.0

def recall_at_k(recommended_items, relevant_items, k):
    recommended_at_k = set(recommended_items[:k])
    relevant_set = set(relevant_items)
    hits = len(recommended_at_k & relevant_set)
    return hits /len(relevant_set) if len(relevant_set) > 0 else 0.0
def hit_rate_at_k(recommended_items, relevant_items, k):
    recommended_at_k = set(recommended_items[:k])
    relevant_set = set(relevant_items)
    return 1.0 if len(recommended_at_k & relevant_set) > 0 else 0.0
def reciprocal_rank(recommended_items, relevant_items):
    relevant_set = set(relevant_items)
    for rank, item in enumerate(recommended_items, 1):
        if item in relevant_set:
            return 1.0 /rank
    return 0.0
def average_precision(recommended_items, relevant_items):
    relevant_set = set(relevant_items)
    hits = 0
    sum_precision = 0.0
    for rank, item in enumerate(recommended_items, 1):
        if item in relevant_set:
            hits += 1
            sum_precision += hits/rank
    return sum_precision /len(relevant_set) if len(relevant_set) > 0 else 0.0

def catalog_coverage(all_recommendations, total_items):
    unique_recommended = set(all_recommendations)
    return len(unique_recommended) / total_items if total_items > 0 else 0.0

def user_coverage(users_with_recommendations, total_users):
    return users_with_recommendations / total_users if total_users > 0 else 0.0
class RecommenderMetrics:
    def __init__(self, relevance_threshold=4):

        self.relevance_threshold = relevance_threshold

    def evaluate_all(self, predictions_df, test_data_df, k_values=[5, 10, 20],
                     num_items=None, num_users=None):
        # Merge predictions with actual ratings
        merged_df = pd.merge(
            test_data_df[['user_id', 'movie_id', 'rating']],predictions_df,on=['user_id', 'movie_id'],how='inner'
        )
        results = {}
# Rating Prediction Metrics
        results['RMSE'] = calculate_rmse(
            merged_df['predicted_rating'].values,merged_df['rating'].values
        )
        results['MAE'] = calculate_mae(
            merged_df['predicted_rating'].values, merged_df['rating'].values
        )
# Ranking Metrics (per user, then averaged)
        user_ids = merged_df['user_id'].unique()
        # Initialize metric accumulators
        metrics_per_k = {k: {
            'ndcg': [], 'precision': [], 'recall': [],'hit_rate': [], 'mrr': [], 'map': []
        } for k in k_values}
        all_recommended_items = []
        users_with_recs = 0
        for user_id in user_ids:
            user_data = merged_df[merged_df['user_id'] == user_id].copy()
            if len(user_data) == 0:
                continue
            users_with_recs += 1
            # Sort by predicted rating (descending) to get ranked recommendations
            user_data = user_data.sort_values('predicted_rating', ascending=False)
            recommended_items = user_data['movie_id'].tolist()
            actual_ratings = user_data['rating'].tolist()

            # Relevant items are those with rating >= threshold
            relevant_items = user_data[
                user_data['rating'] >= self.relevance_threshold
            ]['movie_id'].tolist()
            # Track all recommended items for coverage
            all_recommended_items.extend(recommended_items)

            # Calculate metrics for each K
            for k in k_values:
                # NDCG@K
                relevance_scores = actual_ratings[:k]
                ideal_scores = sorted(actual_ratings, reverse=True)[:k]
                ndcg = ndcg_at_k(relevance_scores, ideal_scores, k)
                metrics_per_k[k]['ndcg'].append(ndcg)
                # Precision@K
                prec = precision_at_k(recommended_items, relevant_items, k)
                metrics_per_k[k]['precision'].append(prec)
                # Recall@K
                rec = recall_at_k(recommended_items, relevant_items, k)
                metrics_per_k[k]['recall'].append(rec)
                # Hit Rate@K
                hr = hit_rate_at_k(recommended_items, relevant_items, k)
                metrics_per_k[k]['hit_rate'].append(hr)
                # MRR (computed on top-K)
                mrr = reciprocal_rank(recommended_items[:k], relevant_items)
                metrics_per_k[k]['mrr'].append(mrr)
                # MAP (computed on top-K)
                ap = average_precision(recommended_items[:k], relevant_items)
                metrics_per_k[k]['map'].append(ap)
        # Average metrics across users
        for k in k_values:
            results[f'NDCG@{k}'] = np.mean(metrics_per_k[k]['ndcg'])
            results[f'Precision@{k}'] = np.mean(metrics_per_k[k]['precision'])
            results[f'Recall@{k}'] = np.mean(metrics_per_k[k]['recall'])
            results[f'HitRate@{k}'] = np.mean(metrics_per_k[k]['hit_rate'])
            results[f'MRR@{k}'] = np.mean(metrics_per_k[k]['mrr'])
            results[f'MAP@{k}'] = np.mean(metrics_per_k[k]['map'])
        # Coverage Metrics
        if num_items is not None:
            results['CatalogCoverage'] = catalog_coverage(all_recommended_items, num_items)
        if num_users is not None:
            results['UserCoverage'] = user_coverage(users_with_recs, num_users)
        return results

    def print_results(self, results, k_values=[5, 10, 20]):
        print('=' * 60)
        print('RECOMMENDER SYSTEM EVALUATION RESULTS')
        # Rating Prediction Metrics
        print('\nRATING PREDICTION METRICS')
        print(f"RMSE:{results['RMSE']:.4f}")
        print(f"MAE: {results['MAE']:.4f}")
        # Ranking Metrics
        print('\n RANKING METRICS')
        # Create a table for ranking metrics
        header = f"{'Metric':<15}" + "".join([f"@{k:<8}" for k in k_values])
        print(header)
        print('-' * len(header))
        for metric in ['NDCG', 'Precision', 'Recall', 'HitRate', 'MRR', 'MAP']:
            row = f"{metric:<15}"
            for k in k_values:
                value = results.get(f'{metric}@{k}', 0)
                row += f"{value:<9.4f}"
            print(row)
        # Coverage Metrics
        if 'CatalogCoverage' in results or 'UserCoverage' in results:
            print('\nCOVERAGE METRICS')
            if 'CatalogCoverage' in results:
                print(f"  Catalog Coverage: {results['CatalogCoverage']:.4f} "
                      f"({results['CatalogCoverage']*100:.2f}%)")
            if 'UserCoverage' in results:
                print(f"  User Coverage:    {results['UserCoverage']:.4f} "
                      f"({results['UserCoverage']*100:.2f}%)")

In [34]:
# Convert to DataFrame
predictions_df = pd.DataFrame(
    test_predictions,
    columns=['user_id', 'movie_id', 'predicted_rating']
)
# Prepare test data DataFrame
test_data_df = filtered_test_data[['user_id', 'movie_id', 'rating']].copy()
# Initialize evaluator (items with rating >= 4 are considered relevant)
evaluator = RecommenderMetrics(relevance_threshold=4)
# Compute all metrics
results = evaluator.evaluate_all(
    predictions_df=predictions_df,test_data_df=test_data_df,k_values=[5, 10, 20], num_items=num_movies,num_users=num_users
)
evaluator.print_results(results, k_values=[5,10, 20])

RECOMMENDER SYSTEM EVALUATION RESULTS

RATING PREDICTION METRICS
RMSE:1.3492
MAE: 1.0093

 RANKING METRICS
Metric         @5       @10      @20      
------------------------------------------
NDCG           0.9249   0.9377   0.9406   
Precision      0.5685   0.3487   0.1890   
Recall         0.9097   0.9747   0.9912   
HitRate        0.9941   0.9941   0.9941   
MRR            0.9916   0.9916   0.9916   
MAP            0.9065   0.9708   0.9870   

COVERAGE METRICS
  Catalog Coverage: 0.6939 (69.39%)
  User Coverage:    0.6990 (69.90%)


# Feature-enhance NMF

In [35]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler, MultiLabelBinarizer
import numpy as np
import pandas as pd
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
# user feature matrix
user_feat_cols = ['num_ratings', 'avg_rating', 'rating_std','avg_popularity_rated', 'avg_rating_most_rated_genre']
# original user_id → encoded user_id
orig_user_ids = user_encoder.inverse_transform(range(num_users))
user_feat_df  = pd.DataFrame({'orig_user_id': orig_user_ids})
user_feat_df  = user_feat_df.merge(
    user_features[['user_id'] + user_feat_cols].rename(columns={'user_id': 'orig_user_id'}),on='orig_user_id', how='left'
).fillna(0)

scaler_u = StandardScaler()
user_feat_matrix = scaler_u.fit_transform(
    user_feat_df[user_feat_cols].values
).astype(np.float32)
print(f'User feature matrix: {user_feat_matrix.shape}')
# anime feature matrix
# Multi-hot genres
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(
    anime_clean['Genres'].str.split(', ')
).astype(np.float32)
# Scale score and popularity
scaler_i = StandardScaler()
anime_extra = scaler_i.fit_transform(
    anime_clean[['Score', 'Popularity']].fillna(0).values
).astype(np.float32)
# Combine genre+Score+Popularity
anime_feat_full = np.hstack([genre_matrix, anime_extra])
# Map original anime_id → encoded movie_id
orig_anime_ids  = movie_encoder.inverse_transform(range(num_movies))
anime_id_to_idx = {aid: i for i, aid in enumerate(anime_clean['anime_id'].values)}
anime_feat_matrix = np.zeros((num_movies, anime_feat_full.shape[1]), dtype=np.float32)
for enc_id, orig_id in enumerate(orig_anime_ids):
    if orig_id in anime_id_to_idx:
        anime_feat_matrix[enc_id] = anime_feat_full[anime_id_to_idx[orig_id]]
print(f'Anime feature matrix: {anime_feat_matrix.shape}')
#Dataset
class AnimeDataset(Dataset):
    def __init__(self, data):
        data = data.reset_index(drop=True)
        self.user_ids  = torch.tensor(data['user_id'].values, dtype=torch.long).to(device)
        self.movie_ids = torch.tensor(data['movie_id'].values, dtype=torch.long).to(device)
        self.ratings   = torch.tensor(data['rating'].values,dtype=torch.float).to(device)
    def __len__(self):
        return len(self.ratings)
    def __getitem__(self, idx):
        return {
            'user_id':self.user_ids[idx],'movie_id': self.movie_ids[idx],'rating': self.ratings[idx]
        }
batch_size= 2048
train_loader = DataLoader(AnimeDataset(train_data), batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=False)
test_loader = DataLoader(AnimeDataset(filtered_test_data), batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)
print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')
#Model
class FeatureEnhancedNCF(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim,
                 user_features, item_features, hidden_dims=[128, 64]):
        super(FeatureEnhancedNCF, self).__init__()
        self.register_buffer('user_features', torch.tensor(user_features, dtype=torch.float32))
        self.register_buffer('item_features', torch.tensor(item_features, dtype=torch.float32))
        user_feat_dim = user_features.shape[1]
        item_feat_dim = item_features.shape[1]
        # GMF embeddings
        self.user_embedding_gmf = nn.Embedding(num_users, embedding_dim)
        self.item_embedding_gmf = nn.Embedding(num_items, embedding_dim)
        # MLP embeddings
        self.user_embedding_mlp = nn.Embedding(num_users, embedding_dim)
        self.item_embedding_mlp = nn.Embedding(num_items, embedding_dim)
        # Feature networks
        self.user_feat_net = nn.Sequential(
            nn.Linear(user_feat_dim, 32), nn.ReLU(), nn.Dropout(0.2),nn.Linear(32, embedding_dim)
        )
        self.item_feat_net = nn.Sequential(
            nn.Linear(item_feat_dim, 32), nn.ReLU(), nn.Dropout(0.2),nn.Linear(32, embedding_dim)
        )
        # Combined MLP
        mlp_input_dim = 4 * embedding_dim
        mlp_layers = []
        prev_dim = mlp_input_dim
        for hidden_dim in hidden_dims:
            mlp_layers.extend([nn.Linear(prev_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.2)])
            prev_dim = hidden_dim
        self.mlp = nn.Sequential(*mlp_layers)
        # Final layer scale to [1, 10]
        final_input_dim = embedding_dim + hidden_dims[-1]
        self.final_layer = nn.Sequential(
            nn.Linear(final_input_dim, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
        self._init_weights()
    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, std=0.01)
            elif isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, user_id, item_id):
        user_emb_gmf = self.user_embedding_gmf(user_id)
        item_emb_gmf = self.item_embedding_gmf(item_id)
        user_emb_mlp = self.user_embedding_mlp(user_id)
        item_emb_mlp = self.item_embedding_mlp(item_id)
        user_feat_emb = self.user_feat_net(self.user_features[user_id])
        item_feat_emb = self.item_feat_net(self.item_features[item_id])
        gmf_output = (user_emb_gmf + user_feat_emb) * (item_emb_gmf + item_feat_emb)
        mlp_input  = torch.cat([user_emb_mlp, item_emb_mlp, user_feat_emb, item_feat_emb], dim=-1)
        mlp_output = self.mlp(mlp_input)
        combined = torch.cat([gmf_output, mlp_output], dim=-1)
        output   = self.final_layer(combined)
        return (output.squeeze() * 9 + 1)# scale [0,1] -> [1,10]
embedding_dim = 16
model     = FeatureEnhancedNCF(
    num_users=num_users,num_items=num_movies,embedding_dim=embedding_dim,
    user_features=user_feat_matrix,item_features=anime_feat_matrix,hidden_dims=[128, 64]
).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(f'Model device: {next(model.parameters()).device}')

#Training
best_loss= float('inf')
patience= 2
patience_counter = 0
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        user_id  = batch['user_id']
        movie_id = batch['movie_id']
        ratings  = batch['rating']
        optimizer.zero_grad()
        loss = criterion(model(user_id, movie_id), ratings)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_train_loss = epoch_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f}', end=' | ')
    model.eval()
    eval_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            user_id  = batch['user_id']
            movie_id = batch['movie_id']
            ratings  = batch['rating']
            eval_loss += criterion(model(user_id, movie_id), ratings).item()
    avg_eval_loss = eval_loss/ len(val_loader)
    print(f'Val Loss: {avg_eval_loss:.4f}')
    if avg_eval_loss < best_loss:
        best_loss        = avg_eval_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_feature_ncf_model.pt')
        print(f'Best model saved (loss: {best_loss:.4f})')
    else:
        patience_counter += 1
        print(f'No improvement. Patience: {patience_counter}/{patience}')
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break
#Final evaluation
print('=' * 50)
model.load_state_dict(torch.load('best_feature_ncf_model.pt', map_location=device))
model.eval()
all_predictions, all_ratings_list = [], []
with torch.no_grad():
    for batch in test_loader:
        user_id  = batch['user_id']
        movie_id = batch['movie_id']
        ratings  = batch['rating']
        outputs  = model(user_id, movie_id)
        all_predictions.extend(outputs.cpu().numpy())
        all_ratings_list.extend(ratings.cpu().numpy())
all_predictions = np.array(all_predictions)
all_ratings_arr = np.array(all_ratings_list)
rmse = np.sqrt(np.mean((all_predictions - all_ratings_arr) ** 2))
mae  = np.mean(np.abs(all_predictions - all_ratings_arr))
print(f'Final RMSE: {rmse:.4f}')
print(f'Final MAE:  {mae:.4f}')

Using device: cuda
User feature matrix: (163930, 5)
Anime feature matrix: (15061, 46)
Train batches: 1796 | Test batches: 224
Model device: cuda:0
Epoch 1/10 | Train Loss: 1.8425 | Val Loss: 1.7080
Best model saved (loss: 1.7080)
Epoch 2/10 | Train Loss: 1.6081 | Val Loss: 1.6721
Best model saved (loss: 1.6721)
Epoch 3/10 | Train Loss: 1.4945 | Val Loss: 1.6636
Best model saved (loss: 1.6636)
Epoch 4/10 | Train Loss: 1.3558 | Val Loss: 1.6838
No improvement. Patience: 1/2
Epoch 5/10 | Train Loss: 1.2105 | Val Loss: 1.7323
No improvement. Patience: 2/2
Early stopping at epoch 5
Final RMSE: 1.2902
Final MAE:  0.9741


In [36]:
def generate_predictions(model, data_loader):
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in data_loader:
            user_id  = batch['user_id']
            movie_id = batch['movie_id']
            outputs  = model(user_id, movie_id)
            predictions.extend(zip(user_id.cpu().numpy(), movie_id.cpu().numpy(), outputs.cpu().numpy()))
    return predictions
test_predictions = generate_predictions(model, test_loader)
enhanced_predictions = test_predictions  # save Feature-Enhanced NCF predictions

In [37]:
# Convert test data to DataFrame for easier manipulation
test_data_df = pd.DataFrame(filtered_test_data, columns=['user_id', 'movie_id', 'rating'])
# Map actual ratings to relevance scores
test_data_df['relevance'] = test_data_df['rating'].apply(lambda x: x if x > 0 else 0)  # Example relevance mapping
test_data_df
len(test_data_df)
len(test_predictions)

458015

In [38]:
from collections import defaultdict
# RATING PREDICTION METRICS
def calculate_rmse(predictions, actuals):
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    return np.sqrt(np.mean((predictions - actuals) ** 2))
def calculate_mae(predictions, actuals):
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    return np.mean(np.abs(predictions - actuals))

# RANKING METRICS
def dcg_at_k(relevance_scores, k):
    relevance_scores = np.asarray(relevance_scores)[:k]
    if relevance_scores.size:
        return np.sum((2**relevance_scores - 1)/np.log2(np.arange(2, relevance_scores.size + 2)))
    return 0.0


def ndcg_at_k(relevance_scores, ideal_relevance_scores, k):
    dcg = dcg_at_k(relevance_scores, k)
    idcg = dcg_at_k(ideal_relevance_scores, k)
    if idcg == 0:
        return 0.0
    return dcg/idcg


def precision_at_k(recommended_items, relevant_items, k):
    recommended_at_k = set(recommended_items[:k])
    relevant_set = set(relevant_items)
    hits = len(recommended_at_k & relevant_set)
    return hits /k if k > 0 else 0.0


def recall_at_k(recommended_items, relevant_items, k):
    recommended_at_k = set(recommended_items[:k])
    relevant_set = set(relevant_items)
    hits = len(recommended_at_k & relevant_set)
    return hits /len(relevant_set) if len(relevant_set) > 0 else 0.0

def hit_rate_at_k(recommended_items, relevant_items, k):
    recommended_at_k = set(recommended_items[:k])
    relevant_set = set(relevant_items)
    return 1.0 if len(recommended_at_k & relevant_set) > 0 else 0.0
def reciprocal_rank(recommended_items, relevant_items):
    relevant_set = set(relevant_items)
    for rank, item in enumerate(recommended_items, 1):
        if item in relevant_set:
            return 1.0 /rank
    return 0.0
def average_precision(recommended_items, relevant_items):
    relevant_set = set(relevant_items)
    hits = 0
    sum_precision = 0.0
    for rank, item in enumerate(recommended_items, 1):
        if item in relevant_set:
            hits += 1
            sum_precision += hits/rank
    return sum_precision /len(relevant_set) if len(relevant_set) > 0 else 0.0

def catalog_coverage(all_recommendations, total_items):
    unique_recommended = set(all_recommendations)
    return len(unique_recommended)/ total_items if total_items > 0 else 0.0

def user_coverage(users_with_recommendations, total_users):
    return users_with_recommendations / total_users if total_users > 0 else 0.0

class RecommenderMetrics:
    def __init__(self, relevance_threshold=4):
        self.relevance_threshold = relevance_threshold
    def evaluate_all(self, predictions_df, test_data_df, k_values=[5, 10, 20],
                     num_items=None, num_users=None):
        # Merge predictions with actual ratings
        merged_df = pd.merge(
            test_data_df[['user_id', 'movie_id', 'rating']],predictions_df,on=['user_id', 'movie_id'],how='inner'
        )
        results = {}
# Rating Prediction Metrics
        results['RMSE'] = calculate_rmse(
            merged_df['predicted_rating'].values, merged_df['rating'].values
        )
        results['MAE'] = calculate_mae(
            merged_df['predicted_rating'].values,merged_df['rating'].values
        )
        user_ids = merged_df['user_id'].unique()
        # Initialize metric accumulators
        metrics_per_k = {k: {
            'ndcg': [], 'precision': [], 'recall': [],'hit_rate': [], 'mrr': [], 'map': []
        } for k in k_values}
        all_recommended_items = []
        users_with_recs = 0
        for user_id in user_ids:
            user_data = merged_df[merged_df['user_id'] == user_id].copy()
            if len(user_data) == 0:
                continue
            users_with_recs += 1
            # Sort by predicted rating (descending) to get ranked recommendations
            user_data = user_data.sort_values('predicted_rating', ascending=False)
            recommended_items = user_data['movie_id'].tolist()
            actual_ratings = user_data['rating'].tolist()
            # Relevant items are those with rating >= threshold
            relevant_items = user_data[user_data['rating'] >= self.relevance_threshold]['movie_id'].tolist()
            # Track all recommended items for coverage
            all_recommended_items.extend(recommended_items)

            # Calculate metrics for each K
            for k in k_values:
                # NDCG@K
                relevance_scores = actual_ratings[:k]
                ideal_scores = sorted(actual_ratings, reverse=True)[:k]
                ndcg = ndcg_at_k(relevance_scores, ideal_scores, k)
                metrics_per_k[k]['ndcg'].append(ndcg)
                # Precision@K
                prec = precision_at_k(recommended_items, relevant_items, k)
                metrics_per_k[k]['precision'].append(prec)
                # Recall@K
                rec = recall_at_k(recommended_items, relevant_items, k)
                metrics_per_k[k]['recall'].append(rec)
                # Hit Rate@K
                hr = hit_rate_at_k(recommended_items, relevant_items, k)
                metrics_per_k[k]['hit_rate'].append(hr)
                # MRR (computed on top-K)
                mrr = reciprocal_rank(recommended_items[:k], relevant_items)
                metrics_per_k[k]['mrr'].append(mrr)
                # MAP (computed on top-K)
                ap = average_precision(recommended_items[:k], relevant_items)
                metrics_per_k[k]['map'].append(ap)

        # Average metrics across users
        for k in k_values:
            results[f'NDCG@{k}'] = np.mean(metrics_per_k[k]['ndcg'])
            results[f'Precision@{k}'] = np.mean(metrics_per_k[k]['precision'])
            results[f'Recall@{k}'] = np.mean(metrics_per_k[k]['recall'])
            results[f'HitRate@{k}'] = np.mean(metrics_per_k[k]['hit_rate'])
            results[f'MRR@{k}'] = np.mean(metrics_per_k[k]['mrr'])
            results[f'MAP@{k}'] = np.mean(metrics_per_k[k]['map'])

        # Coverage Metrics
        if num_items is not None:
            results['CatalogCoverage'] = catalog_coverage(all_recommended_items, num_items)

        if num_users is not None:
            results['UserCoverage'] = user_coverage(users_with_recs, num_users)
        return results
    def print_results(self, results, k_values=[5, 10, 20]):
        print('=' * 60)
        print('RECOMMENDER SYSTEM EVALUATION RESULTS')
        # Rating Prediction Metrics
        print('\nRATING PREDICTION METRICS')
        print(f"  RMSE:{results['RMSE']:.4f}")
        print(f"  MAE:{results['MAE']:.4f}")
        # Ranking Metrics
        print('\n RANKING METRICS')
        # Create a table for ranking metrics
        header = f"{'Metric':<15}" + "".join([f"@{k:<8}" for k in k_values])
        print(header)
        print('-' * len(header))
        for metric in ['NDCG', 'Precision', 'Recall', 'HitRate', 'MRR', 'MAP']:
            row = f"{metric:<15}"
            for k in k_values:
                value = results.get(f'{metric}@{k}', 0)
                row += f"{value:<9.4f}"
            print(row)
        # Coverage Metrics
        if 'CatalogCoverage' in results or 'UserCoverage' in results:
            print('\n COVERAGE METRICS')
            print('-' * 40)
            if 'CatalogCoverage' in results:
                print(f"  Catalog Coverage: {results['CatalogCoverage']:.4f} "
                      f"({results['CatalogCoverage']*100:.2f}%)")
            if 'UserCoverage' in results:
                print(f"  User Coverage:    {results['UserCoverage']:.4f} "
                      f"({results['UserCoverage']*100:.2f}%)")
# Convert to DataFrame
predictions_df = pd.DataFrame(test_predictions,columns=['user_id', 'movie_id', 'predicted_rating'])
# Prepare test data DataFrame
test_data_df = filtered_test_data[['user_id', 'movie_id', 'rating']].copy()
# Initialize evaluator (items with rating >= 4 are considered relevant)
evaluator = RecommenderMetrics(relevance_threshold=4)
results = evaluator.evaluate_all(predictions_df=predictions_df,
    test_data_df=test_data_df,k_values=[5, 10, 20],num_items=num_movies, num_users=num_users
)
evaluator.print_results(results, k_values=[5, 10, 20])

RECOMMENDER SYSTEM EVALUATION RESULTS

RATING PREDICTION METRICS
  RMSE:1.2902
  MAE:0.9741

 RANKING METRICS
Metric         @5       @10      @20      
------------------------------------------
NDCG           0.9297   0.9415   0.9443   
Precision      0.5688   0.3488   0.1890   
Recall         0.9098   0.9748   0.9913   
HitRate        0.9941   0.9941   0.9941   
MRR            0.9919   0.9920   0.9920   
MAP            0.9070   0.9713   0.9875   

 COVERAGE METRICS
----------------------------------------
  Catalog Coverage: 0.6939 (69.39%)
  User Coverage:    0.6990 (69.90%)


# **Analysis by User Segment**

In [39]:
#MF predictions
model = MatrixFactorization(num_users, num_movies, embedding_dim).to(device)
model.load_state_dict(torch.load('best_model.pt', map_location=device))
model.eval()
test_predictions = generate_predictions(model, test_loader)
mf_predictions = test_predictions

#NeuMF predictions
model = NCF(num_users, num_movies, embedding_dim).to(device)
model.load_state_dict(torch.load('best_neumf_model.pt', map_location=device))
model.eval()
test_predictions = generate_predictions(model, test_loader)
neumf_predictions = test_predictions

# Feature-Enhanced NCF predictions
model = FeatureEnhancedNCF(num_users, num_movies, embedding_dim,user_feat_matrix, anime_feat_matrix).to(device)
model.load_state_dict(torch.load('best_feature_ncf_model.pt', map_location=device))
model.eval()
test_predictions = generate_predictions(model, test_loader)
enhanced_predictions = test_predictions

In [40]:
encoded_to_orig = {enc: orig for enc, orig in
                   enumerate(user_encoder.inverse_transform(range(num_users)))}
test_data_df['user_id_orig'] = test_data_df['user_id'].map(encoded_to_orig)
test_data_df['segment'] = test_data_df['user_id_orig'].map(
    user_counts.set_index('user_id')['segment']
)
evaluator = RecommenderMetrics(relevance_threshold=7)
for model_name, preds in [('MF', mf_predictions),('NeuMF', neumf_predictions), ('Feature-Enhanced NCF', enhanced_predictions)]:
    print(f'\n{"#"*50}')
    print(f'MODEL: {model_name}')
    pred_df = pd.DataFrame(preds, columns=['user_id_enc', 'movie_id', 'predicted_rating'])
    pred_df['user_id_orig'] = pred_df['user_id_enc'].map(encoded_to_orig)
    pred_df['segment'] = pred_df['user_id_orig'].map(
        user_counts.set_index('user_id')['segment']
    )
    for segment in ['low', 'medium', 'high']:
        seg_users = pred_df[pred_df['segment'] == segment]['user_id_enc'].unique()
        seg_predictions = pred_df[pred_df['segment'] == segment][
            ['user_id_enc', 'movie_id', 'predicted_rating']
        ].rename(columns={'user_id_enc': 'user_id'})
        seg_test = test_data_df[test_data_df['user_id'].isin(seg_users)][
            ['user_id', 'movie_id', 'rating']
        ]
        if len(seg_predictions) == 0:
            print(f'\nSegment {segment}: No predictions found')
            continue
        print(f'\n{"="*50}')
        print(f'SEGMENT: {segment.upper()} ACTIVITY USERS')
        print(f'Users: {len(seg_users):,} | Predictions: {len(seg_predictions):,}')
        seg_results = evaluator.evaluate_all(
            predictions_df=seg_predictions, test_data_df=seg_test,k_values=[5,10,20],num_items=num_movies,num_users=len(seg_users)
        )
        evaluator.print_results(seg_results, k_values=[5,10,20])


##################################################
MODEL: MF

SEGMENT: LOW ACTIVITY USERS
Users: 28,863 | Predictions: 43,814
RECOMMENDER SYSTEM EVALUATION RESULTS

RATING PREDICTION METRICS
  RMSE:5.9531
  MAE:5.2557

 RANKING METRICS
Metric         @5       @10      @20      
------------------------------------------
NDCG           0.9658   0.9659   0.9659   
Precision      0.2600   0.1301   0.0651   
Recall         0.9025   0.9028   0.9028   
HitRate        0.9028   0.9028   0.9028   
MRR            0.8811   0.8811   0.8811   
MAP            0.8798   0.8800   0.8800   

 COVERAGE METRICS
----------------------------------------
  Catalog Coverage: 0.2631 (26.31%)
  User Coverage:    1.0000 (100.00%)

SEGMENT: MEDIUM ACTIVITY USERS
Users: 66,869 | Predictions: 229,981
RECOMMENDER SYSTEM EVALUATION RESULTS

RATING PREDICTION METRICS
  RMSE:5.6401
  MAE:4.9853

 RANKING METRICS
Metric         @5       @10      @20      
------------------------------------------
NDCG           0.8695

# **Full Catalouge Eval of MF and NeuMF**

In [ ]:
import torch
import numpy as np
import pandas as pd
from collections import defaultdict
encoded_to_orig_user = {enc: orig for enc, orig in enumerate(user_encoder.inverse_transform(range(num_users)))}
encoded_to_orig_anime = {enc: orig for enc, orig in enumerate(movie_encoder.inverse_transform(range(num_movies)))}
# anime_id -> anime name lookup
anime_id_to_name = anime_clean.set_index('anime_id')['Name'].to_dict()
# per-user watched set (encoded ids) — used to mask already-seen anime
train_watched = (train_data.groupby('user_id')['movie_id'].apply(set).to_dict())
# 1. FULL-CATALOG TOP-N GENERATION
@torch.no_grad()
def generate_topN_fullcatalog_mf(model, num_users, num_movies,ntrain_watched, N=10, block_size=512):
    model.eval()
    # Extract embedding weight matrices directly — shape [n, dim]
    U =model.user_embeddings.weight.detach() # [num_users, dim]
    I = model.item_embeddings.weight.detach() # [num_movies, dim]
    all_items = torch.arange(num_movies, device=U.device)
    topN_dict = {}
    for start in range(0, num_users, block_size):
        end = min(start + block_size, num_users)
        u_blk = torch.arange(start, end, device=U.device)
        # [block, dim] @ [dim, num_movies] → [block, num_movies]
        scores = U[u_blk] @ I.T
        # [1, 10] to match model forward()
        scores = torch.clamp(scores, 1, 10)
        # mask already-watched items
        for bi, u in enumerate(u_blk.tolist()):
            seen = train_watched.get(u, set())
            if seen:
                seen_t = torch.tensor(list(seen), dtype=torch.long, device=scores.device)
                scores[bi, seen_t] = -1e9
        topk = torch.topk(scores, k=N, dim=1).indices # [block, N]
        for bi, u in enumerate(u_blk.tolist()):
            topN_dict[u] = topk[bi].cpu().tolist()
    return topN_dict
@torch.no_grad()
def generate_topN_fullcatalog_ncf(model, num_users, num_movies,train_watched, N=10, block_size=256):
    model.eval()
    all_items = torch.arange(num_movies, device=device)
    topN_dict = {}
    for start in range(0, num_users, block_size):
        end = min(start + block_size, num_users)
        u_blk = torch.arange(start, end, device=device)# [B]
        # expand: ech user repeated num_movies times
        u_exp =u_blk.unsqueeze(1).expand(-1, num_movies).reshape(-1) # [B*M]
        i_exp =all_items.unsqueeze(0).expand(end -start, -1).reshape(-1)  # [B*M]
        scores = model(u_exp, i_exp).reshape(end -start,num_movies) # [B, M]
        # mask already-watched items
        for bi, u in enumerate(u_blk.tolist()):
            seen = train_watched.get(u, set())
            if seen:
                seen_t =torch.tensor(list(seen), dtype=torch.long, device=scores.device)
                scores[bi, seen_t] = -1e9
        topk = torch.topk(scores, k=N, dim=1).indices  # [B, N]
        for bi, u in enumerate(u_blk.tolist()):
            topN_dict[u] =topk[bi].cpu().tolist()
    return topN_dict
# 2. CATALOG COVRAGE
def compute_catalog_coverage(topN_dict, num_movies, label=''):
    recommended_items = set()
    for items in topN_dict.values():
        recommended_items.update(items)
    coverage = len(recommended_items) /num_movies
    print(f'[{label}] Catalog Coverage: {coverage:.4f} '
          f'({coverage*100:.2f}%)  '
          f'[{len(recommended_items):,} /{num_movies:,} anime]')
    return coverage
# 3. FULL-CATALOG RANKING METRICS (per segment)
def evaluate_topN_by_segment(topN_dict, filtered_test_data, user_counts,encoded_to_orig_user, k_values=[5, 10, 20],relevance_threshold=7, label=''):
    # Build ground truth: encoded user → set of relevant encoded anime
    ground_truth = defaultdict(set)
    for row in filtered_test_data.itertuples(index=False):
        if row.rating >= relevance_threshold:
            ground_truth[row.user_id].add(row.movie_id)
    # Attach segment to each user
    orig_to_segment = user_counts.set_index('user_id')['segment'].to_dict()
    seg_metrics = defaultdict(lambda: defaultdict(list))
    for enc_user, top_items in topN_dict.items():
        orig_user = encoded_to_orig_user.get(enc_user)
        segment   = orig_to_segment.get(orig_user, 'unknown')
        relevant  = ground_truth.get(enc_user, set())
        if not relevant:
            continue
        for k in k_values:
            recs_k = top_items[:k]
            hits   = [1 if x in relevant else 0 for x in recs_k]
            n_hits = sum(hits)
            n_rel  = len(relevant)
            # Precision@k
            seg_metrics[segment][f'precision@{k}'].append(n_hits/k)
            # Recall@k
            seg_metrics[segment][f'recall@{k}'].append(n_hits/n_rel)
            # HitRate@k
            seg_metrics[segment][f'hitrate@{k}'].append(1.0 if n_hits >0 else 0.0)
            # NDCG@k
            idcg = sum(1.0 /np.log2(i +2) for i in range(min(n_rel,k)))
            dcg  = sum(h /np.log2(i +2) for i, h in enumerate(hits))
            seg_metrics[segment][f'ndcg@{k}'].append(dcg / idcg if idcg > 0 else 0.0)
            # MRR (only for k=max to keep it position-independent)
            if k == max(k_values):
                rr = 0.0
                for rank, item in enumerate(top_items, 1):
                    if item in relevant:
                        rr = 1.0 / rank
                        break
                seg_metrics[segment]['mrr'].append(rr)
            # MAP@k
            prec_sum, pos = 0.0, 0
            for r, h in enumerate(hits, 1):
                if h:
                    pos += 1
                    prec_sum += pos/r
            seg_metrics[segment][f'map@{k}'].append(
                prec_sum / min(n_rel, k) if n_rel > 0 else 0.0
            )
    print(f'\n{"#"*60}')
    print(f'MODEL: {label}  [Full-Catalog Top-N Evaluation]')
    for segment in ['low', 'medium', 'high']:
        m = seg_metrics.get(segment, {})
        if not m:
            continue
        n_users = len(m.get(f'ndcg@{k_values[0]}', []))
        print(f'\n{"="*50}')
        print(f'SEGMENT: {segment.upper()} ACTIVITY USERS  ({n_users:,} users)')
        print(f'  {"Metric":<14}', end='')
        for k in k_values:
            print(f'  @{k:<6}', end='')
        print()
        print(f'  {"-"*40}')
        for metric_base in ['ndcg', 'precision', 'recall', 'hitrate', 'map']:
            print(f'  {metric_base.upper():<14}', end='')
            for k in k_values:
                key  = f'{metric_base}@{k}'
                vals = m.get(key, [])
                print(f'  {np.mean(vals):.4f}', end='')
            print()
        print(f'  {"MRR":<14}  {np.mean(m.get("mrr", [0])):.4f}')
    return seg_metrics

def print_sample_recommendations(topN_dict, encoded_to_orig_user,encoded_to_orig_anime, anime_id_to_name, n_users=3, N=10, label=''):
    print(f'\n{"="*60}')
    print(f'SAMPLE TOP-{N} RECOMMENDATIONS  [{label}]')
    sample_users = list(topN_dict.keys())[:n_users]
    for enc_u in sample_users:
        orig_u   = encoded_to_orig_user.get(enc_u, enc_u)
        top_animes = topN_dict[enc_u]
        print(f'\n  User {orig_u} (encoded {enc_u}):')
        for rank, enc_a in enumerate(top_animes, 1):
            orig_a = encoded_to_orig_anime.get(enc_a, enc_a)
            name   = anime_id_to_name.get(orig_a, f'anime_id={orig_a}')
            print(f'    {rank:>2}. {name}')

# 5. Run all3 models
K_VALUES = [5,10,20]
TOP_N    = 20 # generat top-20 so to evaluate @5, @10, @20
print('=' * 60)
print('FULL-CATALOG TOP-N GENERATION & COVERAGE ANALYSIS')
mf_model = MatrixFactorization(num_users, num_movies, embedding_dim).to(device)
mf_model.load_state_dict(torch.load('best_model.pt', map_location=device))
mf_model.eval()
neumf_model = NCF(num_users, num_movies, embedding_dim).to(device)
neumf_model.load_state_dict(torch.load('best_neumf_model.pt', map_location=device))
neumf_model.eval()

feat_model = FeatureEnhancedNCF(
    num_users=num_users,num_items=num_movies,embedding_dim=embedding_dim,
    user_features=user_feat_matrix,item_features=anime_feat_matrix,hidden_dims=[128, 64]
).to(device)
feat_model.load_state_dict(torch.load('best_feature_ncf_model.pt', map_location=device))
feat_model.eval()
# MF
print('\n[1/3]')
mf_topN = generate_topN_fullcatalog_mf(
    mf_model, num_users, num_movies, train_watched, N=TOP_N
)
compute_catalog_coverage(mf_topN, num_movies, label='MF')
evaluate_topN_by_segment(
    mf_topN, filtered_test_data, user_counts,
    encoded_to_orig_user, k_values=K_VALUES, label='MF'
)
print_sample_recommendations(
    mf_topN, encoded_to_orig_user, encoded_to_orig_anime,
    anime_id_to_name, n_users=3, N=10, label='MF'
)
#NeuM
print('\n[2/3]')
neumf_topN = generate_topN_fullcatalog_ncf(neumf_model, num_users, num_movies, train_watched, N=TOP_N)
compute_catalog_coverage(neumf_topN, num_movies, label='NeuMF')
evaluate_topN_by_segment(
    neumf_topN, filtered_test_data, user_counts,
    encoded_to_orig_user, k_values=K_VALUES, label='NeuMF'
)
print_sample_recommendations(
    neumf_topN, encoded_to_orig_user, encoded_to_orig_anime,
    anime_id_to_name, n_users=3, N=10, label='NeuMF'
)

#Feature-Enhanced NCF
print('\n[3/3] ')
feat_topN = generate_topN_fullcatalog_ncf(
    feat_model, num_users, num_movies, train_watched, N=TOP_N
)
compute_catalog_coverage(feat_topN, num_movies, label='Feature-Enhanced NCF')
evaluate_topN_by_segment(
    feat_topN, filtered_test_data, user_counts,
    encoded_to_orig_user, k_values=K_VALUES, label='Feature-Enhanced NCF'
)
print_sample_recommendations(
    feat_topN, encoded_to_orig_user, encoded_to_orig_anime,
    anime_id_to_name, n_users=3, N=10, label='Feature-Enhanced NCF'
)

print('COVERAGE')
print(f'  {"Model":<25}  {"Items Covered":>14}  {"Coverage %":>11}')
for label, topN in [('MF', mf_topN), ('NeuMF', neumf_topN),('Feature-Enhanced NCF', feat_topN)]:
    covered = len(set(a for items in topN.values() for a in items))
    pct= covered / num_movies * 100
    print(f'  {label:<25}  {covered:>14,}  {pct:>10.2f}%')
print(f'  Total anime in catalog: {num_movies:,}')

FULL-CATALOG TOP-N GENERATION & COVERAGE ANALYSIS

[1/3] Generating top-N for MF (fast: direct embedding multiply)...
[MF] Catalog Coverage: 0.3329 (33.29%)  [5,014 / 15,061 anime]

############################################################
MODEL: MF  [Full-Catalog Top-N Evaluation]

SEGMENT: LOW ACTIVITY USERS  (26,057 users)
  Metric          @5       @10      @20    
  ----------------------------------------
  NDCG            0.0028  0.0053  0.0076
  PRECISION       0.0013  0.0017  0.0014
  RECALL          0.0047  0.0114  0.0199
  HITRATE         0.0066  0.0165  0.0282
  MAP             0.0019  0.0029  0.0034
  MRR             0.0049

SEGMENT: MEDIUM ACTIVITY USERS  (63,171 users)
  Metric          @5       @10      @20    
  ----------------------------------------
  NDCG            0.0022  0.0040  0.0057
  PRECISION       0.0016  0.0019  0.0017
  RECALL          0.0029  0.0071  0.0123
  HITRATE         0.0077  0.0191  0.0323
  MAP             0.0013  0.0018  0.0022
  MRR       

# **Post-hoc Adjustment With Penalty Weights**

In [ ]:
# TOP-N WITH POPULARITY PENALTY RE-RANKING
# comparison of coverage and metrics for lambda = 0.0, 0.3, 0.5, 0.7
import torch
import numpy as np
import pandas as pd
from collections import defaultdict
# 1. Popularity tensor (once, reused across all models/lambdas)
popularity_counts = train_data.groupby('movie_id').size()
pop_array = np.array(
    [popularity_counts.get(i, 0) for i in range(num_movies)],
    dtype=np.float32
)
pop_array = pop_array /pop_array.max()# normalise to [0, 1]
pop_tensor = torch.tensor(pop_array, device=device) # shape [num_movies]
print(f'Popularity tensor built: {num_movies:,} anime')
print(f'Most popular anime (encoded id {pop_array.argmax()}): '
      f'count={popularity_counts.max():,}')
# 2. GENERATE FUNCTIONS WITH OPTIONAL PENALTY
@torch.no_grad()
def generate_topN_mf_penalty(model, num_users, num_movies,train_watched, pop_tensor,N=20, block_size=512, lambda_penalty=0.3):
    model.eval()
    U = model.user_embeddings.weight.detach()
    I = model.item_embeddings.weight.detach()
    topN_dict = {}
    for start in range(0, num_users, block_size):
        end   = min(start + block_size, num_users)
        u_blk = torch.arange(start, end, device=U.device)

        scores = U[u_blk] @ I.T# [block, num_movies]
        scores = torch.clamp(scores, 1, 10)
        # popularity penalty
        if lambda_penalty >0:
            scores = scores - lambda_penalty*pop_tensor.unsqueeze(0)
        # mask already-watched
        for bi, u in enumerate(u_blk.tolist()):
            seen = train_watched.get(u, set())
            if seen:
                seen_t = torch.tensor(list(seen), dtype=torch.long, device=scores.device)
                scores[bi, seen_t] = -1e9
        topk = torch.topk(scores, k=N, dim=1).indices
        for bi, u in enumerate(u_blk.tolist()):
            topN_dict[u] = topk[bi].cpu().tolist()
    return topN_dict
@torch.no_grad()
def generate_topN_ncf_penalty(model, num_users, num_movies,train_watched, pop_tensor,N=20, block_size=256, lambda_penalty=0.3):
    model.eval()
    all_items = torch.arange(num_movies, device=device)
    topN_dict = {}
    for start in range(0, num_users, block_size):
        end   = min(start + block_size, num_users)
        u_blk = torch.arange(start, end, device=device)

        u_exp = u_blk.unsqueeze(1).expand(-1, num_movies).reshape(-1)
        i_exp = all_items.unsqueeze(0).expand(end - start, -1).reshape(-1)

        scores = model(u_exp, i_exp).reshape(end - start, num_movies)

        # popularity penalt
        if lambda_penalty > 0:
            scores = scores - lambda_penalty * pop_tensor.unsqueeze(0)

        # mask already-watched
        for bi, u in enumerate(u_blk.tolist()):
            seen = train_watched.get(u, set())
            if seen:
                seen_t = torch.tensor(list(seen), dtype=torch.long,
                                      device=scores.device)
                scores[bi, seen_t] = -1e9

        topk = torch.topk(scores, k=N, dim=1).indices
        for bi, u in enumerate(u_blk.tolist()):
            topN_dict[u] = topk[bi].cpu().tolist()

        if start % (block_size * 20) == 0:
            print(f'  ... {end:,}/{num_users:,} users', end='\r')
    print()
    return topN_dict
# 3. HELPER: COVERAGE
def get_coverage(topN_dict, num_movies):
    covered = set(a for items in topN_dict.values() for a in items)
    return len(covered), len(covered) / num_movies * 100
# 4. HELPER: SEGMENT NDCG@10 (quick scalar for comparison table)
def quick_ndcg10_by_segment(topN_dict, filtered_test_data, user_counts,
                             encoded_to_orig_user, relevance_threshold=7):
    ground_truth = defaultdict(set)
    for row in filtered_test_data.itertuples(index=False):
        if row.rating >= relevance_threshold:
            ground_truth[row.user_id].add(row.movie_id)
    orig_to_segment = user_counts.set_index('user_id')['segment'].to_dict()
    seg_ndcg = defaultdict(list)
    for enc_user, top_items in topN_dict.items():
        orig_user = encoded_to_orig_user.get(enc_user)
        segment = orig_to_segment.get(orig_user, 'unknown')
        relevant = ground_truth.get(enc_user, set())
        if not relevant:
            continue
        k    = 10
        hits = [1 if x in relevant else 0 for x in top_items[:k]]
        idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
        dcg  = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
        seg_ndcg[segment].append(dcg / idcg if idcg > 0 else 0.0)

    return {seg: float(np.mean(vals)) for seg, vals in seg_ndcg.items()}
LAMBDA_VALUES = [0.0, 0.3, 0.5, 0.7]
TOP_N         = 20
K_NDCG        = 10
# store results for comparison table
results = []

print('=' * 70)
print('POPULARITY PENALTY SWEEP')
print('=' * 70)

for lambda_val in LAMBDA_VALUES:
    print(f'\nlambda = {lambda_val}')
    #MF
    mf_pen = generate_topN_mf_penalty(
        mf_model, num_users, num_movies, train_watched,pop_tensor, N=TOP_N, lambda_penalty=lambda_val
    )
    mf_cov, mf_cov_pct = get_coverage(mf_pen, num_movies)
    mf_ndcg = quick_ndcg10_by_segment(
        mf_pen, filtered_test_data, user_counts, encoded_to_orig_user
    )
    print(f'[MF]  Coverage: {mf_cov_pct:.2f}% ({mf_cov:,} anime) | '
          f'NDCG@10 low={mf_ndcg.get("low",0):.4f} '
          f'med={mf_ndcg.get("medium",0):.4f} '
          f'high={mf_ndcg.get("high",0):.4f}')

    #NeuMF
    neumf_pen = generate_topN_ncf_penalty(
        neumf_model, num_users, num_movies, train_watched,pop_tensor, N=TOP_N, lambda_penalty=lambda_val
    )
    neumf_cov, neumf_cov_pct = get_coverage(neumf_pen, num_movies)
    neumf_ndcg = quick_ndcg10_by_segment(
        neumf_pen, filtered_test_data, user_counts, encoded_to_orig_user
    )
    print(f'  [NeuMF] Coverage: {neumf_cov_pct:.2f}% ({neumf_cov:,} anime) | '
          f'NDCG@10 low={neumf_ndcg.get("low",0):.4f} '
          f'med={neumf_ndcg.get("medium",0):.4f} '
          f'high={neumf_ndcg.get("high",0):.4f}')

    #Feature-Enhanced NCF
    feat_pen = generate_topN_ncf_penalty(
        feat_model, num_users, num_movies, train_watched,pop_tensor, N=TOP_N, lambda_penalty=lambda_val
    )
    feat_cov, feat_cov_pct = get_coverage(feat_pen, num_movies)
    feat_ndcg = quick_ndcg10_by_segment(
        feat_pen, filtered_test_data, user_counts, encoded_to_orig_user
    )
    print(f'[Feat-NCF] Coverage: {feat_cov_pct:.2f}% ({feat_cov:,} anime) | '
          f'NDCG@10 low={feat_ndcg.get("low",0):.4f} '
          f'med={feat_ndcg.get("medium",0):.4f} '
          f'high={feat_ndcg.get("high",0):.4f}')
    results.append({
        'lam':lambda_val,
        'mf_coverage':mf_cov_pct,
        'mf_ndcg_low': mf_ndcg.get('low', 0),
        'mf_ndcg_med': mf_ndcg.get('medium', 0),
        'mf_ndcg_high':mf_ndcg.get('high', 0),
        'neumf_coverage':neumf_cov_pct,
        'neumf_ndcg_low':neumf_ndcg.get('low', 0),
        'neumf_ndcg_med':neumf_ndcg.get('medium', 0),
        'neumf_ndcg_high':neumf_ndcg.get('high', 0),
        'feat_coverage': feat_cov_pct,
        'feat_ndcg_low': feat_ndcg.get('low', 0),
        'feat_ndcg_med': feat_ndcg.get('medium', 0),
        'feat_ndcg_high':feat_ndcg.get('high', 0),
    })

# 6. Comparison
results_df = pd.DataFrame(results)
print('\n CATALOG COVERAGE (%)')
print(f'  {"lambda":<10} {"MF":>10} {"NeuMF":>10} {"Feat-NCF":>10}')
print(f'  {"-"*44}')
for _, row in results_df.iterrows():
    label = f'λ={row.lam:.1f}' + (' [baseline]' if row.lam == 0.0 else '')
    print(f'  {label:<18} {row.mf_coverage:>8.2f}%  '
          f'{row.neumf_coverage:>8.2f}%  '
          f'{row.feat_coverage:>8.2f}%')

print('\nNDCG@10 — LOW ACTIVITY USERS')
print(f'  {"lambda":<10} {"MF":>10} {"NeuMF":>10} {"Feat-NCF":>10}')
print(f'  {"-"*44}')
for _, row in results_df.iterrows():
    label = f'λ={row.lam:.1f}' + (' [baseline]' if row.lam == 0.0 else '')
    print(f'  {label:<18} {row.mf_ndcg_low:>10.4f}  '
          f'{row.neumf_ndcg_low:>8.4f}  '
          f'{row.feat_ndcg_low:>8.4f}')

print('\nNDCG@10 — MEDIUM ACTIVITY USERS')
print(f'  {"lambda":<10} {"MF":>10} {"NeuMF":>10} {"Feat-NCF":>10}')
print(f'  {"-"*44}')
for _, row in results_df.iterrows():
    label = f'λ={row.lam:.1f}' + (' [baseline]' if row.lam == 0.0 else '')
    print(f'  {label:<18} {row.mf_ndcg_med:>10.4f}  '
          f'{row.neumf_ndcg_med:>8.4f}  '
          f'{row.feat_ndcg_med:>8.4f}')

print('\nNDCG@10 — HIGH ACTIVITY USERS')
print(f'  {"lambda":<10} {"MF":>10} {"NeuMF":>10} {"Feat-NCF":>10}')
print(f'  {"-"*44}')
for _, row in results_df.iterrows():
    label = f'λ={row.lam:.1f}' + (' [baseline]' if row.lam == 0.0 else '')
    print(f'  {label:<18} {row.mf_ndcg_high:>10.4f}  '
          f'{row.neumf_ndcg_high:>8.4f}  '
          f'{row.feat_ndcg_high:>8.4f}')
# 7. SAMPLE RECOMMENDATIONS COMPARISON (best lambda vs baseline, Feat-NCF)
# pick best lambda = 0.5 for sample comparison
best_lambda = 0.5
print(f'\n\n{"="*70}')
print(f'SAMPLE RECOMMENDATIONS: λ=0.0 (baseline) vs λ={best_lambda} (re-ranked)')
# regenerate feat at best lambda for sample
feat_best = generate_topN_ncf_penalty(
    feat_model, num_users, num_movies, train_watched,pop_tensor, N=20, lambda_penalty=best_lambda
)
print(f'\n  {"Rank":<6} {"λ=0.0 (no penalty)":<45} {"λ="+str(best_lambda)+" (penalised)":<45}')
print(f'  {"-"*95}')

baseline_recs = feat_topN.get(0, [])[:20]   # from previous cell
penalised_recs = feat_best.get(0, [])[:20]

for rank, (base_enc, pen_enc) in enumerate(zip(baseline_recs, penalised_recs), 1):
    base_orig = encoded_to_orig_anime.get(base_enc, base_enc)
    pen_orig= encoded_to_orig_anime.get(pen_enc, pen_enc)
    base_name= anime_id_to_name.get(base_orig, f'id={base_orig}')[:42]
    pen_name = anime_id_to_name.get(pen_orig,  f'id={pen_orig}')[:42]
    print(f'{rank:<6} {base_name:<45} {pen_name:<45}')

Popularity tensor built: 15,061 anime
Most popular anime (encoded id 1393): count=15,703
POPULARITY PENALTY SWEEP

── lambda = 0.0 ──────────────────────────────────────────
  [MF] generating...
  [MF]  Coverage: 33.29% (5,014 anime) | NDCG@10 low=0.0053 med=0.0040 high=0.0039
  [NeuMF] generating...
  ... 163,930/163,930 users
  [NeuMF] Coverage: 7.58% (1,141 anime) | NDCG@10 low=0.0049 med=0.0047 high=0.0047
  [Feat-NCF] generating...
  ... 163,930/163,930 users
  [Feat-NCF] Coverage: 7.41% (1,116 anime) | NDCG@10 low=0.0079 med=0.0083 high=0.0090

── lambda = 0.3 ──────────────────────────────────────────
  [MF] generating...
  [MF]  Coverage: 91.61% (13,797 anime) | NDCG@10 low=0.0000 med=0.0000 high=0.0000
  [NeuMF] generating...
  ... 163,930/163,930 users
  [NeuMF] Coverage: 8.95% (1,348 anime) | NDCG@10 low=0.0017 med=0.0023 high=0.0028
  [Feat-NCF] generating...
  ... 163,930/163,930 users
  [Feat-NCF] Coverage: 8.14% (1,226 anime) | NDCG@10 low=0.0038 med=0.0046 high=0.0060



# **Two tower using Reviews**

In [ ]:
reviews = pd.read_csv('/content/drive/My Drive/Colab Notebooks/jikan_crawled_reviews_with_userid.csv')
if 'sypnopsis' in anime_clean.columns:
    anime_clean = anime_clean.rename(columns={'sypnopsis': 'Synopsis'})
if 'Synopsis' in anime_clean.columns:
    anime_clean['Synopsis'] = anime_clean['Synopsis'].fillna('')

reviews_clean = reviews.copy()
reviews_clean.columns = [col.strip().replace(' ', '_') for col in reviews_clean.columns]

print(f"Original reviews: {len(reviews_clean):,}")
print("Review columns:", reviews_clean.columns.tolist())

# Convert key columns
reviews_clean['review_date_utc'] = pd.to_datetime(reviews_clean['review_date_utc'], utc=True, errors='coerce')
reviews_clean['review_score'] = pd.to_numeric(reviews_clean['review_score'], errors='coerce')

# Keep only needed columns if they exist
needed_cols = ['user_id', 'anime_id', 'review_date_utc', 'review_score', 'review_text']
existing_needed_cols = [c for c in needed_cols if c in reviews_clean.columns]
reviews_clean = reviews_clean[existing_needed_cols].copy()

# Drop missing critical fields
reviews_clean = reviews_clean.dropna(subset=['user_id', 'anime_id', 'review_date_utc', 'review_score']).copy()

# Remove duplicates
reviews_clean = reviews_clean.drop_duplicates(subset=['user_id', 'anime_id', 'review_date_utc']).reset_index(drop=True)

print(f"Cleaned reviews: {len(reviews_clean):,}")

data = reviews_clean.merge(
    anime_clean[['anime_id', 'Name', 'Genres', 'Type', 'Episodes', 'Score', 'Synopsis']]
    if 'Synopsis' in anime_clean.columns
    else anime_clean[['anime_id', 'Name', 'Genres', 'Type', 'Episodes', 'Score']],
    on='anime_id',
    how='left'
)

data = data.dropna(subset=['user_id', 'anime_id']).copy()
data = data.sort_values(['user_id', 'review_date_utc']).reset_index(drop=True)

print(f"Merged review-anime rows: {len(data):,}")

# Binary label for retrieval task
data['label'] = (data['review_score'] >= 7).astype(int)

print("Label distribution:")
print(data['label'].value_counts(dropna=False))
print(data['label'].value_counts(normalize=True, dropna=False))

# Keep only users with enough interaction history
user_counts = data.groupby('user_id').size()
valid_users = user_counts[user_counts >= 5].index

data = data[data['user_id'].isin(valid_users)].copy()
data = data.sort_values(['user_id', 'review_date_utc']).reset_index(drop=True)

print(f"Users kept: {data['user_id'].nunique():,}")
print(f"Rows kept: {len(data):,}")

Original reviews: 19,870
Review columns: ['user_id', 'username', 'review_id', 'anime_id', 'entry_title', 'review_date_utc', 'review_text', 'entry_url', 'review_url', 'review_score', 'is_spoiler', 'is_preliminary', 'crawl_timestamp_utc']
Cleaned reviews: 19,870
Merged review-anime rows: 19,870
Label distribution:
label
1    14899
0     4971
Name: count, dtype: int64
label
1    0.749824
0    0.250176
Name: proportion, dtype: float64
Users kept: 822
Rows kept: 12,158


In [ ]:
# Ensure that for each users, earlier interactions are in train, later interactions in val/test
def chrono_split_user(df, train_frac=0.8, val_frac=0.1):
    df = df.sort_values('review_date_utc').copy()
    n = len(df)

    train_end = max(1, int(n * train_frac))
    val_end = max(train_end + 1, int(n * (train_frac + val_frac))) if n >= 3 else n

    df['split'] = 'test'
    df.iloc[:train_end, df.columns.get_loc('split')] = 'train'
    df.iloc[train_end:val_end, df.columns.get_loc('split')] = 'val'
    return df

data = (
    data.groupby('user_id', group_keys=False)
    .apply(chrono_split_user)
    .reset_index(drop=True)
)

print(data['split'].value_counts())
print(data['split'].value_counts(normalize=True))

split
train    9419
test     1375
val      1364
Name: count, dtype: int64
split
train    0.774716
test     0.113094
val      0.112190
Name: proportion, dtype: float64


/tmp/ipykernel_11324/247250488.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(chrono_split_user)


In [ ]:
# Build leakage-safe past review text history
data = data.sort_values(['user_id', 'review_date_utc']).copy()

# Number of past interactions before the current one
data['hist_count'] = data.groupby('user_id').cumcount()

# Past score features only
data['hist_mean_score'] = (
    data.groupby('user_id')['review_score']
    .transform(lambda s: s.expanding().mean().shift(1))
)

data['hist_std_score'] = (
    data.groupby('user_id')['review_score']
    .transform(lambda s: s.expanding().std().shift(1))
)

data['hist_mean_score'] = data['hist_mean_score'].fillna(0)
data['hist_std_score'] = data['hist_std_score'].fillna(0)

def build_past_review_text(df):
    df = df.sort_values('review_date_utc').copy()
    history = []
    past_texts = []

    for txt in df['review_text'].fillna('').astype(str):
        past_texts.append(' '.join(history))   # only past reviews
        history.append(txt)                    # add current review after

    df['past_review_text'] = past_texts
    return df

data = (
    data.groupby('user_id', group_keys=False)
    .apply(build_past_review_text)
    .reset_index(drop=True)
)

print(
    data[['user_id', 'anime_id', 'review_score', 'hist_count',
          'hist_mean_score', 'hist_std_score', 'past_review_text', 'label', 'split']].head(10)
)

# Build historical genre preference features
data['Genres'] = data['Genres'].fillna('')

data['genre_list'] = data['Genres'].apply(
    lambda x: [g.strip() for g in str(x).split(',') if g.strip() != '']
)

all_genres = sorted(set(g for genres in data['genre_list'] for g in genres))
genre_to_idx = {g: i for i, g in enumerate(all_genres)}

print(f"Number of genres: {len(all_genres)}")
print(all_genres[:10])

   user_id  anime_id  review_score  hist_count  hist_mean_score  \
0        1         1            10           0         0.000000   
1        1       263            10           1        10.000000   
2        1       565             8           2        10.000000   
3        1       853             8           3         9.333333   
4        1       849            10           4         9.000000   
5        1         7             8           5         9.200000   
6        1         6            10           6         9.000000   
7        1        25             8           7         9.142857   
8        1      1519             6           8         9.000000   
9        1      1210             8           9         8.666667   

   hist_std_score                                   past_review_text  label  \
0        0.000000                                                         1   
1        0.000000  Cowboy Bebop is an episodic series. By episodi...      1   
2        0.000000  Cowboy

/tmp/ipykernel_11324/2753792287.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_past_review_text)


In [ ]:
# 2.3 Create past-only user genre history vectors

def build_past_genre_matrix(df, genre_to_idx):
    df = df.sort_values('review_date_utc').copy()
    num_genres = len(genre_to_idx)

    running = np.zeros(num_genres, dtype=np.float32)
    hist_vectors = []

    for _, row in df.iterrows():
        hist_vectors.append(running.copy())

        for g in row['genre_list']:
            if g in genre_to_idx:
                running[genre_to_idx[g]] += 1.0

    cols = [f'user_genre_{g}' for g in genre_to_idx.keys()]
    hist_df = pd.DataFrame(hist_vectors, columns=cols, index=df.index)
    return hist_df

genre_feature_parts = []

for user_id, df_user in data.groupby('user_id'):
    hist_df = build_past_genre_matrix(df_user, genre_to_idx)
    genre_feature_parts.append(hist_df)

genre_feature_df = pd.concat(genre_feature_parts).sort_index()

genre_cols = genre_feature_df.columns.tolist()

# Drop old genre-history columns if they already exist from a previous run
old_genre_cols = [c for c in data.columns if c.startswith('user_genre_')]
if old_genre_cols:
    data = data.drop(columns=old_genre_cols)

# join back to data
data = data.join(genre_feature_df)

row_sums = data[genre_cols].sum(axis=1)
row_sums = row_sums.replace(0, 1)

data[genre_cols] = data[genre_cols].div(row_sums, axis=0)

print(data[['user_id', 'anime_id', 'genre_list'] + genre_cols[:5]].head())
print(data[genre_cols].shape)


   user_id  anime_id                                         genre_list  \
0        1         1  [Action, Adventure, Comedy, Drama, Sci-Fi, Space]   
1        1       263                   [Comedy, Sports, Drama, Shounen]   
2        1       565  [Action, Military, Sci-Fi, Adventure, Historic...   
3        1       853           [Comedy, Harem, Romance, School, Shoujo]   
4        1       849  [Comedy, Mystery, Parody, School, Sci-Fi, Slic...   

   user_genre_Action  user_genre_Adventure  user_genre_Cars  \
0           0.000000              0.000000              0.0   
1           0.166667              0.166667              0.0   
2           0.100000              0.100000              0.0   
3           0.125000              0.125000              0.0   
4           0.095238              0.095238              0.0   

   user_genre_Comedy  user_genre_Dementia  
0           0.000000                  0.0  
1           0.166667                  0.0  
2           0.200000                  

In [ ]:
# Create one unique row per anime
item_df = data[['anime_id', 'Synopsis', 'genre_list', 'Type', 'Episodes', 'Name']].drop_duplicates('anime_id').copy()

# Fill any remaining missing values
item_df['Synopsis'] = item_df['Synopsis'].fillna('')
item_df['Type'] = item_df['Type'].fillna('Unknown')
item_df['Episodes'] = pd.to_numeric(item_df['Episodes'], errors='coerce').fillna(0)

# TF-IDF on synopsis
tfidf = TfidfVectorizer(max_features= 100, stop_words='english')
item_tfidf = tfidf.fit_transform(item_df['Synopsis']).astype(np.float32)

# Multi-hot encode genres
mlb = MultiLabelBinarizer()
item_genres = mlb.fit_transform(item_df['genre_list'])

# One-hot encode anime type
type_dummies = pd.get_dummies(item_df['Type'], prefix='type')

# Scale episodes
scaler = StandardScaler()
item_episodes = scaler.fit_transform(item_df[['Episodes']])

# Combine dense item features
item_dense = np.hstack([
    item_genres.astype(np.float32),
    type_dummies.values.astype(np.float32),
    item_episodes.astype(np.float32)
])

# Mapping from anime_id to item row
anime_id_to_row = dict(zip(item_df['anime_id'].values, np.arange(len(item_df))))
row_to_anime_id = dict(zip(np.arange(len(item_df)), item_df['anime_id'].values))

print("item_df shape:", item_df.shape)
print("item_tfidf shape:", item_tfidf.shape)
print("item_dense shape:", item_dense.shape)

item_df shape: (5078, 6)
item_tfidf shape: (5078, 100)
item_dense shape: (5078, 51)


In [ ]:
# 2.4b Add safe past-only user history feature: mean episodes watched

data['item_episodes'] = pd.to_numeric(data['Episodes'], errors='coerce').fillna(0)

data['hist_mean_episodes'] = (
    data.groupby('user_id')['item_episodes']
    .transform(lambda s: s.expanding().mean().shift(1))
    .fillna(0)
)

print(data[['user_id', 'anime_id', 'item_episodes', 'hist_mean_episodes']].head(10))

   user_id  anime_id  item_episodes  hist_mean_episodes
0        1         1           26.0            0.000000
1        1       263           75.0           26.000000
2        1       565            1.0           50.500000
3        1       853           26.0           34.000000
4        1       849           14.0           32.000000
5        1         7           26.0           28.400000
6        1         6           26.0           28.000000
7        1        25           24.0           27.714286
8        1      1519           12.0           27.250000
9        1      1210           24.0           25.555556


In [ ]:
# 2.5 Keep only rows usable for the two-tower model
model_data = data[data['hist_count'] > 0].copy()
model_data = model_data[model_data['anime_id'].isin(anime_id_to_row)].copy()

model_data = model_data.sort_values(['user_id', 'review_date_utc']).reset_index(drop=True)

print("model_data shape:", model_data.shape)
print(model_data['split'].value_counts())
print("Unique users:", model_data['user_id'].nunique())
print("Unique anime:", model_data['anime_id'].nunique())

# Build TF-IDF features from PAST review text only
train_mask = model_data['split'] == 'train'

user_text_vectorizer = TfidfVectorizer(
    max_features=300,
    stop_words='english'
)

user_text_train = user_text_vectorizer.fit_transform(
    model_data.loc[train_mask, 'past_review_text']
).astype(np.float32)

user_text_other = user_text_vectorizer.transform(
    model_data.loc[~train_mask, 'past_review_text']
).astype(np.float32)

# Create full matrix aligned to model_data row order
from scipy.sparse import vstack

user_text_features = np.zeros((len(model_data), len(user_text_vectorizer.get_feature_names_out())), dtype=np.float32)
user_text_features[train_mask.values] = user_text_train.toarray()
user_text_features[~train_mask.values] = user_text_other.toarray()

print("user_text_features shape:", user_text_features.shape)

# Encode users into 0..N-1
user_encoder = LabelEncoder()
model_data['user_idx'] = user_encoder.fit_transform(model_data['user_id'])

# Map anime_id to item row index
model_data['item_idx'] = model_data['anime_id'].map(anime_id_to_row)

# Drop any rows that somehow failed item mapping
model_data = model_data.dropna(subset=['item_idx']).copy()
model_data['item_idx'] = model_data['item_idx'].astype(int)

num_users = model_data['user_idx'].nunique()
num_items = len(item_df)



print("model_data shape:", model_data.shape)
print("num_users:", num_users)
print("num_items:", num_items)

model_data[['user_id', 'user_idx', 'anime_id', 'item_idx', 'split']].head()

model_data shape: (11336, 63)
split
train    8597
test     1375
val      1364
Name: count, dtype: int64
Unique users: 822
Unique anime: 4950
user_text_features shape: (11336, 300)
model_data shape: (11336, 65)
num_users: 822
num_items: 5078


,user_id,user_idx,anime_id,item_idx,split
0,1,0,263,1,train
1,1,0,565,2,train
2,1,0,853,3,train
3,1,0,849,4,train
4,1,0,7,5,train


In [ ]:
# 2.7 Create positive and negative pairs for two-tower training
# positive pair = actual user interaction
# negative pair = anime the user had not seen before that time

rng = np.random.default_rng(42)

all_item_ids = set(item_df['anime_id'].tolist())
model_data = model_data.sort_values(['user_id', 'review_date_utc']).copy()

user_seen_so_far = {}
rows = []

for idx, row in model_data.iterrows():
    u = row['user_id']
    i = row['anime_id']

    if u not in user_seen_so_far:
        user_seen_so_far[u] = set()

    seen_before = user_seen_so_far[u].copy()

    # only rows with review_score >= 7 become positive examples
    if row['label'] == 1 and len(seen_before) > 0:
        rows.append({
            'user_idx': row['user_idx'],
            'item_idx': row['item_idx'],
            'label': 1,
            'split': row['split'],
            'row_id': idx
        })

        candidate_neg = list(all_item_ids - seen_before - {i})

        if len(candidate_neg) > 0:
            neg_items = rng.choice(
                candidate_neg,
                size=min(9, len(candidate_neg)),
                replace=False
            )

            for neg_anime in neg_items:
                rows.append({
                    'user_idx': row['user_idx'],
                    'item_idx': anime_id_to_row[neg_anime],
                    'label': 0,
                    'split': row['split'],
                    'row_id': idx
                })

    # update history only after current row
    user_seen_so_far[u].add(i)

pair_df = pd.DataFrame(rows)

print("pair_df shape:", pair_df.shape)
print(pair_df['label'].value_counts())
print(pair_df['split'].value_counts())
pair_df.head()

pair_df shape: (75450, 5)
label
0    67905
1     7545
Name: count, dtype: int64
split
train    55940
test      9810
val       9700
Name: count, dtype: int64


,user_idx,item_idx,label,split,row_id
0,0,2,1,train,1
1,0,627,0,train,1
2,0,4110,0,train,1
3,0,4350,0,train,1
4,0,4857,0,train,1


In [ ]:
# 2.8 Attach leakage-safe user text history features to pair_df
# and also merge them back into model_data for recommendation use

# Reset clean row_id aligned with user_text_features
model_data = model_data.reset_index(drop=True).copy()
model_data['row_id'] = model_data.index

# Remove any old user_text_* columns first to avoid _x / _y duplication
old_model_text_cols = [c for c in model_data.columns if c.startswith('user_text_')]
if old_model_text_cols:
    model_data = model_data.drop(columns=old_model_text_cols)

old_pair_text_cols = [c for c in pair_df.columns if c.startswith('user_text_')]
if old_pair_text_cols:
    pair_df = pair_df.drop(columns=old_pair_text_cols)

# Build one text-feature row per original interaction row
row_feature_df = pd.DataFrame(user_text_features)
row_feature_df.columns = [f'user_text_{i}' for i in range(row_feature_df.shape[1])]
row_feature_df['row_id'] = model_data['row_id'].values

# Merge text features into model_data
model_data = model_data.merge(row_feature_df, on='row_id', how='left')

# Columns to bring from model_data into pair_df
base_hist_cols = ['row_id', 'hist_count', 'hist_mean_score', 'hist_std_score']

# include optional columns if they exist
optional_hist_cols = [c for c in ['hist_mean_episodes'] if c in model_data.columns]
genre_hist_cols = [c for c in model_data.columns if c.startswith('genre_hist_')]
user_text_cols = [c for c in model_data.columns if c.startswith('user_text_')]

cols_to_merge = base_hist_cols + optional_hist_cols + genre_hist_cols + user_text_cols

# Remove any already-existing duplicate cols in pair_df except row_id
dup_cols = [c for c in cols_to_merge if c in pair_df.columns and c != 'row_id']
if dup_cols:
    pair_df = pair_df.drop(columns=dup_cols)

# Merge all user-side features into pair_df
pair_df = pair_df.merge(
    model_data[cols_to_merge],
    on='row_id',
    how='left'
)

print("pair_df shape after merge:", pair_df.shape)
print("model_data shape after merge:", model_data.shape)
print("Number of user text features:", len(user_text_cols))
print("Missing text features in pair_df:", pair_df[user_text_cols].isna().sum().sum())
print("Missing text features in model_data:", model_data[user_text_cols].isna().sum().sum())

display_cols = ['row_id', 'user_idx', 'item_idx', 'label', 'split'] + user_text_cols[:5]
print(pair_df[display_cols].head())

pair_df shape after merge: (75450, 309)
model_data shape after merge: (11336, 366)
Number of user text features: 300
Missing text features in pair_df: 0
Missing text features in model_data: 0
   row_id  user_idx  item_idx  label  split  user_text_0  user_text_1  \
0       1         0         2      1  train          0.0     0.092577   
1       1         0       627      0  train          0.0     0.092577   
2       1         0      4110      0  train          0.0     0.092577   
3       1         0      4350      0  train          0.0     0.092577   
4       1         0      4857      0  train          0.0     0.092577   

   user_text_2  user_text_3  user_text_4  
0          0.0          0.0          0.0  
1          0.0          0.0          0.0  
2          0.0          0.0          0.0  
3          0.0          0.0          0.0  
4          0.0          0.0          0.0  


In [ ]:
# 2.9 Split pair_df into train / val / test

train_df = pair_df[pair_df['split'] == 'train'].reset_index(drop=True)
val_df   = pair_df[pair_df['split'] == 'val'].reset_index(drop=True)
test_df  = pair_df[pair_df['split'] == 'test'].reset_index(drop=True)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

print(train_df['label'].value_counts())
print(val_df['label'].value_counts())
print(test_df['label'].value_counts())

train: (55940, 309)
val: (9700, 309)
test: (9810, 309)
label
0    50346
1     5594
Name: count, dtype: int64
label
0    8730
1     970
Name: count, dtype: int64
label
0    8829
1     981
Name: count, dtype: int64


In [ ]:
# 3.0 Build PyTorch datasets and dataloaders

import torch
from torch.utils.data import Dataset, DataLoader

base_user_hist_cols = ['hist_count', 'hist_mean_score', 'hist_std_score']

# include optional numeric history features if they exist
optional_cols = []
for c in ['hist_mean_episodes']:
    if c in pair_df.columns:
        optional_cols.append(c)

# include genre history if it exists
genre_cols_existing = [c for c in pair_df.columns if c.startswith('genre_hist_')]

# final combined feature set
user_hist_cols = base_user_hist_cols + optional_cols + genre_cols_existing + user_text_cols

print("Total user feature columns:", len(user_hist_cols))
print(user_hist_cols[:10])

class TwoTowerDataset(Dataset):
    def __init__(self, df, user_hist_cols):
        self.df = df.reset_index(drop=True)

        self.user_idx = self.df['user_idx'].values.astype(np.int64)
        self.item_idx = self.df['item_idx'].values.astype(np.int64)
        self.user_hist = self.df[user_hist_cols].values.astype(np.float32)
        self.labels = self.df['label'].values.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            'user_idx': torch.tensor(self.user_idx[idx], dtype=torch.long),
            'item_idx': torch.tensor(self.item_idx[idx], dtype=torch.long),
            'user_hist': torch.tensor(self.user_hist[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

train_dataset = TwoTowerDataset(train_df, user_hist_cols)
val_dataset   = TwoTowerDataset(val_df, user_hist_cols)
test_dataset  = TwoTowerDataset(test_df, user_hist_cols)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=256, shuffle=False)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Total user feature columns: 304
['hist_count', 'hist_mean_score', 'hist_std_score', 'hist_mean_episodes', 'user_text_0', 'user_text_1', 'user_text_2', 'user_text_3', 'user_text_4', 'user_text_5']
Train batches: 219
Val batches: 38
Test batches: 39


In [ ]:
# 3.1 Convert item features to tensors

item_tfidf_dense = item_tfidf.toarray().astype(np.float32)
item_dense = item_dense.astype(np.float32)

item_tfidf_tensor = torch.tensor(item_tfidf_dense, dtype=torch.float32)
item_dense_tensor = torch.tensor(item_dense, dtype=torch.float32)

print("item_tfidf_tensor:", item_tfidf_tensor.shape)
print("item_dense_tensor:", item_dense_tensor.shape)

item_tfidf_tensor: torch.Size([5078, 100])
item_dense_tensor: torch.Size([5078, 51])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TwoTowerModel(nn.Module):
    def __init__(self, num_users, user_hist_dim, item_tfidf_dim, item_dense_dim, emb_dim=32):
        super().__init__()

        self.user_embedding = nn.Embedding(num_users, 64)

        self.user_mlp = nn.Sequential(
            nn.Linear(64 + user_hist_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, emb_dim)
        )

        self.item_mlp = nn.Sequential(
            nn.Linear(item_tfidf_dim + item_dense_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, emb_dim)
        )

    def forward(self, user_idx, user_hist, item_idx, item_tfidf_tensor, item_dense_tensor):
        user_emb = self.user_embedding(user_idx)
        user_input = torch.cat([user_emb, user_hist], dim=1)
        user_vec = self.user_mlp(user_input)

        item_input = torch.cat([
            item_tfidf_tensor[item_idx],
            item_dense_tensor[item_idx]
        ], dim=1)
        item_vec = self.item_mlp(item_input)

        score = 3.2 * (F.normalize(user_vec, dim=1) * F.normalize(item_vec, dim=1)).sum(dim=1)
        return score

In [ ]:
# 3.3 Initialize model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = TwoTowerModel(
    num_users=num_users,
    user_hist_dim=len(user_hist_cols),
    item_tfidf_dim=item_tfidf_tensor.shape[1],
    item_dense_dim=item_dense_tensor.shape[1],
    emb_dim=64
).to(device)

item_tfidf_tensor = item_tfidf_tensor.to(device)
item_dense_tensor = item_dense_tensor.to(device)

print(model)
print("Device:", device)

TwoTowerModel(
  (user_embedding): Embedding(822, 64)
  (user_mlp): Sequential(
    (0): Linear(in_features=368, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=64, bias=True)
  )
  (item_mlp): Sequential(
    (0): Linear(in_features=151, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
  )
)
Device: cuda


In [ ]:
# 3.4 Train the two-tower model

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

best_val_loss = float('inf')
patience = 5
patience_counter = 0
num_epochs = 20

for epoch in range(num_epochs):
    # -------- train --------
    model.train()
    train_loss = 0.0

    for batch in train_loader:
        user_idx = batch['user_idx'].to(device)
        item_idx = batch['item_idx'].to(device)
        user_hist = batch['user_hist'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()

        logits = model(
            user_idx=user_idx,
            user_hist=user_hist,
            item_idx=item_idx,
            item_tfidf_tensor=item_tfidf_tensor,
            item_dense_tensor=item_dense_tensor
        )

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= max(1, len(train_loader))

    # -------- validation --------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            user_idx = batch['user_idx'].to(device)
            item_idx = batch['item_idx'].to(device)
            user_hist = batch['user_hist'].to(device)
            labels = batch['label'].to(device)

            logits = model(
                user_idx=user_idx,
                user_hist=user_hist,
                item_idx=item_idx,
                item_tfidf_tensor=item_tfidf_tensor,
                item_dense_tensor=item_dense_tensor
            )

            loss = criterion(logits, labels)
            val_loss += loss.item()

    val_loss /= max(1, len(val_loader))

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    # -------- early stopping --------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_two_tower.pt')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

Epoch 1: train_loss=0.3160, val_loss=0.3096
Epoch 2: train_loss=0.3007, val_loss=0.3034
Epoch 3: train_loss=0.2950, val_loss=0.3003
Epoch 4: train_loss=0.2898, val_loss=0.2979
Epoch 5: train_loss=0.2856, val_loss=0.2963
Epoch 6: train_loss=0.2838, val_loss=0.2969
Epoch 7: train_loss=0.2801, val_loss=0.2961
Epoch 8: train_loss=0.2777, val_loss=0.2948
Epoch 9: train_loss=0.2758, val_loss=0.2959
Epoch 10: train_loss=0.2742, val_loss=0.2953
Epoch 11: train_loss=0.2725, val_loss=0.2937
Epoch 12: train_loss=0.2708, val_loss=0.2953
Epoch 13: train_loss=0.2685, val_loss=0.2933
Epoch 14: train_loss=0.2669, val_loss=0.2964
Epoch 15: train_loss=0.2658, val_loss=0.2946
Epoch 16: train_loss=0.2641, val_loss=0.2949
Epoch 17: train_loss=0.2631, val_loss=0.2932
Epoch 18: train_loss=0.2616, val_loss=0.2925
Epoch 19: train_loss=0.2613, val_loss=0.2923
Epoch 20: train_loss=0.2590, val_loss=0.2936


In [ ]:
# 3.5 Evaluate on test set

from sklearn.metrics import roc_auc_score, accuracy_score

model.load_state_dict(torch.load('best_two_tower.pt', map_location=device))
model.eval()

criterion = nn.BCEWithLogitsLoss()

all_logits = []
all_probs = []
all_labels = []
test_loss = 0.0
num_batches = 0

with torch.no_grad():
    for batch in test_loader:
        user_idx = batch['user_idx'].to(device)
        item_idx = batch['item_idx'].to(device)
        user_hist = batch['user_hist'].to(device)
        labels = batch['label'].to(device)

        logits = model(
            user_idx=user_idx,
            user_hist=user_hist,
            item_idx=item_idx,
            item_tfidf_tensor=item_tfidf_tensor,
            item_dense_tensor=item_dense_tensor
        )

        loss = criterion(logits, labels)
        test_loss += loss.item()
        num_batches += 1

        probs = torch.sigmoid(logits)

        all_logits.extend(logits.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_logits = np.array(all_logits)
all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

preds = (all_probs >= 0.5).astype(int)
test_bce = test_loss / max(1, num_batches)

print("Test BCE:", round(test_bce, 4))
print("Test AUC:", round(roc_auc_score(all_labels, all_probs), 4))
print("Test Accuracy:", round(accuracy_score(all_labels, preds), 4))

Test BCE: 0.3021
Test AUC: 0.6911
Test Accuracy: 0.8994


In [ ]:
# 3.6 Ranking metric helpers

import numpy as np
import pandas as pd

def precision_at_k(relevances, k):
    relevances = np.asarray(relevances)[:k]
    if len(relevances) == 0:
        return 0.0
    return np.sum(relevances) / k

def hitrate_at_k(relevances, k):
    relevances = np.asarray(relevances)[:k]
    return 1.0 if np.sum(relevances) > 0 else 0.0

def mrr_at_k(relevances, k):
    relevances = np.asarray(relevances)[:k]
    for idx, rel in enumerate(relevances, start=1):
        if rel == 1:
            return 1.0 / idx
    return 0.0

def ap_at_k(relevances, k):
    relevances = np.asarray(relevances)[:k]
    num_relevant = np.sum(relevances)
    if num_relevant == 0:
        return 0.0

    precisions = []
    hits = 0
    for i, rel in enumerate(relevances, start=1):
        if rel == 1:
            hits += 1
            precisions.append(hits / i)

    return np.mean(precisions) if precisions else 0.0

def dcg_at_k(relevances, k):
    relevances = np.asarray(relevances)[:k]
    if len(relevances) == 0:
        return 0.0
    return np.sum((2**relevances - 1) / np.log2(np.arange(2, len(relevances) + 2)))

def ndcg_at_k(relevances, k):
    actual_dcg = dcg_at_k(relevances, k)
    ideal_relevances = sorted(relevances, reverse=True)
    ideal_dcg = dcg_at_k(ideal_relevances, k)
    if ideal_dcg == 0:
        return 0.0
    return actual_dcg / ideal_dcg

def recall_at_k(relevances, k):
    relevances = np.asarray(relevances)
    total_relevant = np.sum(relevances)
    if total_relevant == 0:
        return 0.0
    return np.sum(relevances[:k]) / total_relevant

In [ ]:
# 3.7 Score every row in test_df for ranking evaluation

model.load_state_dict(torch.load('best_two_tower.pt', map_location=device))
model.eval()

test_eval_df = test_df.copy().reset_index(drop=True)

test_dataset_eval = TwoTowerDataset(test_eval_df, user_hist_cols)
test_loader_eval = DataLoader(test_dataset_eval, batch_size=256, shuffle=False)

all_logits = []

with torch.no_grad():
    for batch in test_loader_eval:
        user_idx = batch['user_idx'].to(device)
        item_idx = batch['item_idx'].to(device)
        user_hist = batch['user_hist'].to(device)

        logits = model(
            user_idx=user_idx,
            user_hist=user_hist,
            item_idx=item_idx,
            item_tfidf_tensor=item_tfidf_tensor,
            item_dense_tensor=item_dense_tensor
        )

        all_logits.extend(logits.cpu().numpy())

test_eval_df['score'] = np.array(all_logits)

print(test_eval_df[['user_idx', 'item_idx', 'label', 'score']].head())
print(test_eval_df.shape)

   user_idx  item_idx  label     score
0         0        15      1 -0.329079
1         0      4917      0 -2.267239
2         0       546      0 -3.145695
3         0      1443      0 -2.602694
4         0      3726      0 -2.914929
(9810, 310)


In [ ]:
# 3.8 Check ranking groups

print("Unique ranking groups in test:", test_eval_df['row_id'].nunique())
print(test_eval_df.groupby('row_id').size().value_counts().sort_index())

Unique ranking groups in test: 981
10    981
Name: count, dtype: int64


In [ ]:
# 3.9 Compute sampled test metrics  on test set

def evaluate_ranking_metrics(
    df,
    group_col='row_id',
    label_col='label',
    score_col='score',
    ks=[1, 2, 3]
):
    results = {}
    grouped = df.groupby(group_col)

    for k in ks:
        precisions = []
        recalls = []
        hitrates = []
        mrrs = []
        maps = []
        ndcgs = []

        for _, group in grouped:
            group_sorted = group.sort_values(score_col, ascending=False)
            relevances = group_sorted[label_col].tolist()

            precisions.append(precision_at_k(relevances, k))
            recalls.append(recall_at_k(relevances, k))
            hitrates.append(hitrate_at_k(relevances, k))
            mrrs.append(mrr_at_k(relevances, k))
            maps.append(ap_at_k(relevances, k))
            ndcgs.append(ndcg_at_k(relevances, k))

        results[f'Precision@{k}'] = np.mean(precisions)
        results[f'Recall@{k}'] = np.mean(recalls)
        results[f'HitRate@{k}'] = np.mean(hitrates)
        results[f'MRR@{k}'] = np.mean(mrrs)
        results[f'MAP@{k}'] = np.mean(maps)
        results[f'NDCG@{k}'] = np.mean(ndcgs)

    return results

ranking_results = evaluate_ranking_metrics(
    test_eval_df,
    group_col='row_id',
    label_col='label',
    score_col='score',
    ks=[1, 2, 3]
)

print("Ranking metrics on sampled test groups:")
for metric, value in ranking_results.items():
    print(f"{metric}: {value:.4f}")

Ranking metrics on sampled test groups:
Precision@1: 0.2905
Recall@1: 0.2905
HitRate@1: 0.2905
MRR@1: 0.2905
MAP@1: 0.2905
NDCG@1: 0.2905
Precision@2: 0.2380
Recall@2: 0.4760
HitRate@2: 0.4760
MRR@2: 0.3833
MAP@2: 0.3833
NDCG@2: 0.4076
Precision@3: 0.1903
Recall@3: 0.5708
HitRate@3: 0.5708
MRR@3: 0.4149
MAP@3: 0.4149
NDCG@3: 0.4550


In [ ]:
# 3.10 Recommend Top-N titles for one user

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd

def recommend_top_n_for_user(
    user_id,
    model,
    model_data,
    item_df,
    item_tfidf_tensor,
    item_dense_tensor,
    user_hist_cols,
    top_n=10,
    exclude_seen=True,
    device='cpu'
):
    model.eval()

    # Get this user's rows in chronological order
    user_rows = model_data[model_data['user_id'] == user_id].sort_values('review_date_utc')

    if len(user_rows) == 0:
        raise ValueError(f"user_id {user_id} not found in model_data")

    # Use the latest available user state
    latest_row = user_rows.iloc[-1]

    user_idx = int(latest_row['user_idx'])
    user_hist = latest_row[user_hist_cols].to_numpy(dtype=np.float32)

    user_idx_tensor = torch.tensor([user_idx], dtype=torch.long, device=device)
    user_hist_tensor = torch.tensor(user_hist, dtype=torch.float32, device=device).unsqueeze(0)

    with torch.no_grad():
        # User vector
        user_emb = model.user_embedding(user_idx_tensor)
        user_input = torch.cat([user_emb, user_hist_tensor], dim=1)
        user_vec = model.user_mlp(user_input)
        user_vec = F.normalize(user_vec, dim=1)

        # Item vectors for all anime
        item_input = torch.cat([item_tfidf_tensor, item_dense_tensor], dim=1)
        item_vecs = model.item_mlp(item_input)
        item_vecs = F.normalize(item_vecs, dim=1)

        # Similarity scores
        scores = 3.2 * torch.matmul(item_vecs, user_vec.T).squeeze(1)

    rec_df = item_df[['anime_id', 'Name']].copy()
    rec_df['score'] = scores.cpu().numpy()

    if exclude_seen:
        seen_anime = set(user_rows['anime_id'].tolist())
        rec_df = rec_df[~rec_df['anime_id'].isin(seen_anime)].copy()

    rec_df = rec_df.sort_values('score', ascending=False).head(top_n).reset_index(drop=True)
    return rec_df

# Change User_id value and rerun code chunk to see reccomendations for different users
sample_user_id = model_data['user_id'].iloc[1503]

top10_recs = recommend_top_n_for_user(
    user_id=sample_user_id,
    model=model,
    model_data=model_data,
    item_df=item_df,
    item_tfidf_tensor=item_tfidf_tensor,
    item_dense_tensor=item_dense_tensor,
    user_hist_cols=user_hist_cols,
    top_n=10,
    exclude_seen=True,
    device=device
)

print(f"Top 10 recommendations for user_id = {sample_user_id}")
print(top10_recs)

Top 10 recommendations for user_id = 14494
   anime_id                                   Name     score
0      2921                        Ashita no Joe 2 -1.082963
1     34798                             Yuru Camp△ -1.130529
2       357           Bokusatsu Tenshi Dokuro-chan -1.154051
3     33352                      Violet Evergarden -1.185147
4     31240  Re:Zero kara Hajimeru Isekai Seikatsu -1.239070
5     39017                           Kyokou Suiri -1.246890
6       475        Hotori: Tada Saiwai wo Koinegau -1.254255
7     40839                   Kanojo, Okarishimasu -1.266528
8     36633                        Date A Live III -1.306575
9       684                  Tenshi no Shippo Chu! -1.307108
